# Tool-Using Agent — Build & Demo Walkthrough

This notebook builds understanding of the agent piece by piece, then runs it
end-to-end against the **real, running local LLM** (Ollama + `qwen2.5:7b-instruct`)
and the **real** calculator / web search / weather tools — nothing here is mocked.

For test coverage and the actual reusable code, see `app/` and `tests/`. This
notebook is the "why and how it works" companion to that code, not a
replacement for it.

**Sections:**
1. Why does an LLM need tools at all?
2. The tools, tested individually
3. The ReAct prompt the agent is built on
4. Walking through one reasoning step by hand
5. The full agent loop, end-to-end, on three example questions
6. The calculator's safety boundary
7. Calling the agent through the deployed FastAPI service

## 0. Setup

In [1]:
# Make sure we're importing the project's own `app` package, not something
# else on the path, regardless of where Jupyter was launched from.
import os, sys

PROJECT_ROOT = os.path.abspath(".")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)

Project root: /Users/tiashaverse/Tool-Using-Agent


In [2]:
import requests

from app import agent
from app.tools import calculator, weather, web_search, get_tool_descriptions, run_tool
from app.config import settings

print("Model:", settings.ollama_model)
print("Ollama host:", settings.ollama_host)

Model: qwen2.5:7b-instruct
Ollama host: http://localhost:11434


/Users/tiashaverse/Tool-Using-Agent/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 1. Why does an LLM need tools at all?

LLMs predict the *most plausible next token*, not the *correct* one. They have
no internal calculator - so for arithmetic, "plausible-looking" and "correct"
can quietly diverge, especially as numbers get bigger. Let's see this directly,
by asking the raw model a multiplication with no tool involved.

In [3]:
# Ask the model directly - same chat API the agent uses internally, but with
# no tool-calling instructions and no verification step.
correct_answer = 837 * 464
print("Actual answer (Python):", correct_answer)

raw_reply = agent._call_ollama([
    {"role": "user", "content": "What is 837 multiplied by 464? Reply with only the number, no explanation."}
])
print("Model's raw answer:     ", raw_reply.strip())

Actual answer (Python): 388368


Model's raw answer:      385368


If the model's raw answer above doesn't exactly match Python's answer, that's
the whole motivation for this project in one cell: the model *sounds* confident
either way, but only one of these answers is actually correct. Handing the
arithmetic to a real calculator tool removes the guesswork entirely - see
Section 5 for the same question asked *through* the agent instead.

## 2. The tools, tested individually

Each tool is a plain Python function - no LLM involved yet. We call them directly first so it's clear exactly what the agent will be calling later.

### 2.1 Calculator

In [4]:
result = calculator.calculate("(45 * 12) + 7")
print("(45 * 12) + 7 =", result)

(45 * 12) + 7 = 547


### 2.2 Weather (real call to the free Open-Meteo API)

In [5]:
print(weather.get_weather("Dublin, Ireland"))

Current weather in Dublin, Ireland: 18.2°C, wind 14.8 km/h, Overcast.


### 2.3 Web search (real call to DuckDuckGo)

In [6]:
print(web_search.search_web("python programming language", max_results=2))

No search results found for 'python programming language'.


**Note on search reliability:** DuckDuckGo's free search backend (no API key
needed) is noticeably less consistent than a paid search API - the same-looking
query can sometimes return zero results. That's a real, observed limitation of
this tool, not a bug in the code around it; Section 5.3 shows the agent handling
that gracefully by falling back to its own knowledge. See `INTERVIEW_PREP_README.md`
Section 3.3 for the full discussion of this tradeoff.

### 2.4 The tool registry

The agent never imports these three modules directly - it only talks to the registry in `app/tools/__init__.py`, which is what turns "add a new tool" into a one-line change anywhere else in the codebase.

In [7]:
print(get_tool_descriptions())

1. calculator: Evaluate a basic arithmetic expression. Input must be a single math expression using only numbers and + - * / // % ** ( ). Example input: '(340 + 12) * 3 / 4'
2. web_search: Search the web for current information (news, facts, definitions, anything not in the model's own knowledge). Input is a plain search query string, e.g. 'population of Iceland 2025'.
3. weather: Get the current weather for a place. Input is just a place name, e.g. 'Tokyo' or 'Dublin, Ireland'.


## 3. The ReAct prompt

This is the entire "brain" of the agent: a system prompt that lists the
available tools and demands a strict output format, so a handful of regexes
can reliably pull the model's decision back out of plain text.

In [8]:
print(agent.SYSTEM_PROMPT_TEMPLATE.format(tool_descriptions=get_tool_descriptions()))

You are a helpful assistant that answers questions by reasoning step by step and using tools when you need information you don't already know.

You have access to the following tools:
1. calculator: Evaluate a basic arithmetic expression. Input must be a single math expression using only numbers and + - * / // % ** ( ). Example input: '(340 + 12) * 3 / 4'
2. web_search: Search the web for current information (news, facts, definitions, anything not in the model's own knowledge). Input is a plain search query string, e.g. 'population of Iceland 2025'.
3. weather: Get the current weather for a place. Input is just a place name, e.g. 'Tokyo' or 'Dublin, Ireland'.

Respond using EXACTLY this format, one field per line, and stop after either
"Action Input" or "Final Answer":

Thought: <your reasoning about what to do next>
Action: <one tool name, exactly as listed above>
Action Input: <the input to give that tool>

...or, once you know the answer...

Thought: <your reasoning about why you no

## 4. Walking through one reasoning step by hand

Before jumping to the full `run_agent()` loop (Section 5), let's manually do
what that loop does internally, one step at a time, so nothing is hidden.

In [9]:
system_prompt = agent.SYSTEM_PROMPT_TEMPLATE.format(tool_descriptions=get_tool_descriptions())

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "What is (45 * 12) + 7?"},
]

# Step 1: ask the model what to do.
reply = agent._call_ollama(messages)
print("--- Model's raw reply ---")
print(reply)

--- Model's raw reply ---
Thought: I need to calculate the expression (45 * 12) + 7.
Action: calculator
Action Input: (45 * 12) + 7


In [10]:
# Step 2: parse that reply into a structured step.
step = agent._parse_response(reply)
print("Thought:     ", step.thought)
print("Action:      ", step.action)
print("Action Input:", step.action_input)

Thought:      I need to calculate the expression (45 * 12) + 7.
Action:       calculator
Action Input: (45 * 12) + 7


In [11]:
# Step 3: actually run the tool the model asked for, and see the Observation
# it will be given next.
observation = run_tool(step.action, step.action_input)
print("Observation:", observation)

# This Observation gets appended to `messages` as a new turn, and we'd loop
# back to Step 1 - which is exactly what run_agent() automates for us.

Observation: 547


## 5. The full agent loop, end-to-end

Now the automated version: `run_agent()` repeats the Reason -> Act -> Observe
cycle from Section 4 until the model gives a `Final Answer`, or hits the
iteration limit (`settings.max_iterations`). Three example questions below,
chosen to exercise each tool plus the "no tool needed" and "tool comes back
empty" cases.

In [12]:
def show_trace(question):
    result = agent.run_agent(question)
    print(f"Q: {question}")
    for i, step in enumerate(result.steps, start=1):
        print(f"  Step {i}: action={step.action!r} input={step.action_input!r}")
        if step.observation:
            print(f"           observation={step.observation!r}")
    print(f"ANSWER: {result.answer}")
    print(f"(hit_max_iterations={result.hit_max_iterations})")
    print("-" * 70)
    return result

### 5.1 A question that needs the calculator

In [13]:
_ = show_trace("What is (45 * 12) + 7?")

Q: What is (45 * 12) + 7?
  Step 1: action='calculator' input='(45 * 12) + 7'
           observation='547'
  Step 2: action='final_answer' input='547'
ANSWER: 547
(hit_max_iterations=False)
----------------------------------------------------------------------


### 5.2 A question that needs the weather tool

In [14]:
_ = show_trace("What is the current weather in Dublin, Ireland?")

Q: What is the current weather in Dublin, Ireland?
  Step 1: action='weather' input='Dublin, Ireland'
           observation='Current weather in Dublin, Ireland: 18.2°C, wind 14.8 km/h, Overcast.'
  Step 2: action='final_answer' input='The current weather in Dublin, Ireland is 18.2°C with wind at 14.8 km/h and it is overcast.'
ANSWER: The current weather in Dublin, Ireland is 18.2°C with wind at 14.8 km/h and it is overcast.
(hit_max_iterations=False)
----------------------------------------------------------------------


### 5.3 A question where search comes back empty, and the model falls back to its own knowledge

In [15]:
_ = show_trace("What is the capital of Portugal?")

Q: What is the capital of Portugal?
  Step 1: action='web_search' input='capital of Portugal'
           observation="No search results found for 'capital of Portugal'."
  Step 2: action='final_answer' input='The capital of Portugal is Lisbon.'
ANSWER: The capital of Portugal is Lisbon.
(hit_max_iterations=False)
----------------------------------------------------------------------


Notice in the trace above: the model *tried* to verify its answer with
`web_search`, got `"No search results found..."` back as the Observation, and
still recovered by falling back to its own knowledge for the final answer -
instead of getting stuck or hallucinating a different answer. That graceful
degradation is a direct result of feeding the tool's real output back to the
model rather than assuming the tool always succeeds.

## 6. The calculator's safety boundary

The calculator tool parses expressions with Python's `ast` module and only
executes a whitelisted set of arithmetic operators - it never calls Python's
`eval()` on the raw string. Here's what happens if we (or a misbehaving model)
try to sneak arbitrary code through the "Action Input" channel.

In [16]:
try:
    calculator.calculate("__import__('os').system('echo this should never run')")
except ValueError as exc:
    print("Rejected safely:", exc)

Rejected safely: Unsupported expression element: Call


## 7. Calling the agent through the deployed FastAPI service

This is what actually gets deployed - the same `run_agent()` function from
Section 5, wrapped behind a REST endpoint. We use FastAPI's `TestClient` here
so the notebook doesn't need a separately-running server, but this sends a
real HTTP request through the real FastAPI app object, request validation
included.

In [17]:
from fastapi.testclient import TestClient
from app.main import app as fastapi_app

client = TestClient(fastapi_app)

response = client.post("/agent", json={"question": "What is (45 * 12) + 7?"})
print("HTTP status:", response.status_code)
response.json()

HTTP status: 200


{'answer': 'The result of (45 * 12) + 7 is 547.',
 'steps': [{'thought': 'I need to calculate the result of the expression.',
   'action': 'calculator',
   'action_input': '(45 * 12) + 7',
   'observation': '547'},
  {'thought': '',
   'action': 'final_answer',
   'action_input': 'The result of (45 * 12) + 7 is 547.',
   'observation': None}],
 'hit_max_iterations': False}

To run this for real over the network instead (e.g. from `curl` or another
service), see the "Running it" section of `README.md` - either
`uvicorn app.main:app --reload` locally, or `docker compose up --build`.

## Where to go next

- `README.md` - quick overview and setup instructions (GitHub-facing).
- `INTERVIEW_PREP_README.md` - the detailed design rationale, tradeoffs, and
  likely interview questions for every decision made in this notebook.
- `tests/` - the pytest suite that keeps the deterministic parts of this
  (parsing, tool safety, the API contract) verified without needing a live
  model for every test run.